In [2]:
# ============================================================
# OOD EVALUATION — NOTEBOOK ĐỘC LẬP
# Tự dựng lại model + load checkpoint từ Drive + đánh giá OOD
# Chỉ cần: checkpoint đã lưu trên Drive (best_mit1003.pth / best_salicon.pth)
# ============================================================
import subprocess, sys

def pip_install(pkg):
    r = subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'],
                       capture_output=True, text=True)
    print(f'{"✅" if r.returncode==0 else "❌"} {pkg}')
    if r.returncode != 0: print(r.stderr[-300:])

pip_install('pysaliency')
pip_install('boltons')
pip_install('efficientnet_pytorch')

# Octave để pysaliency parse fixation .mat của một số dataset
subprocess.run('apt-get install -y octave octave-image octave-statistics liboctave-dev -q'.split())

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

CFG = {
    'batch_size': 2, 'num_workers': 0,
    'downsample': 2, 'readout_factor': 16, 'saliency_map_factor': 2,
    'initial_sigma': 8.0, 'n_components': 10,
    'ds_mit': 2.0,
    'local_ckpt': '/content/checkpoints',
    'drive_ckpt': '/content/drive/MyDrive/Data PBL4/checkpoints',
}
import os
os.makedirs(CFG['local_ckpt'], exist_ok=True)
print('✅ Cell 1 OK')

✅ pysaliency
✅ boltons
✅ efficientnet_pytorch
Device: cpu
✅ Cell 1 OK


In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Checkpoints trên Drive:')
ck = CFG['drive_ckpt']
print(os.listdir(ck) if os.path.exists(ck) else '❌ Không thấy thư mục checkpoints')

Mounted at /content/drive
Checkpoints trên Drive:
['salicon_epoch1.pth', 'salicon_epoch2.pth', 'salicon_epoch3.pth', 'salicon_epoch4.pth', 'salicon_epoch5.pth', 'best_salicon.pth', 'salicon_epoch6.pth', 'salicon_epoch7.pth', 'salicon_epoch8.pth', 'mit_epoch1.pth', 'mit_epoch2.pth', 'mit_epoch3.pth', 'best_mit1003.pth', 'mit_epoch4.pth', 'mit_epoch5.pth', 'mit_epoch6.pth', 'mit_epoch7.pth', 'mit_epoch8.pth', 'training_curves.png', 'deepgaze2e_final_weights.pth']


In [5]:
import math, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict

def gaussian_filter_1d(tensor, dim, sigma, truncate=4, kernel_size=None,
                        padding_mode='replicate', padding_value=0.0):
    sigma = torch.as_tensor(sigma, device=tensor.device, dtype=tensor.dtype)
    if kernel_size is None:
        kernel_size = torch.as_tensor(2*torch.ceil(truncate*sigma)+1,
                                      device=tensor.device, dtype=torch.int64)
    kernel_size = kernel_size.detach()
    kernel_size_int = kernel_size.detach().cpu().numpy()
    mean = (torch.as_tensor(kernel_size, dtype=tensor.dtype)-1)/2
    grid = (torch.arange(kernel_size, device=tensor.device)-mean).view(1,1,-1).detach()
    src = tensor.shape
    tensor = torch.movedim(tensor, dim, len(src)-1)
    dls = tensor.shape
    tensor = tensor.reshape(-1,1,src[dim])
    pad = (math.ceil((kernel_size_int-1)/2),)*2
    t_ = F.pad(tensor, pad, padding_mode, padding_value)
    kernel = torch.exp(-0.5*(grid/sigma)**2); kernel = kernel/kernel.sum()
    t_ = F.conv1d(t_, kernel); t_ = t_.view(dls)
    return torch.movedim(t_, len(src)-1, dim)

class GaussianFilterNd(nn.Module):
    def __init__(self, dims, sigma, truncate=4, trainable=False,
                 padding_mode='replicate', padding_value=0.0, kernel_size=None):
        super().__init__()
        self.dims=dims
        self.sigma=nn.Parameter(torch.tensor(sigma,dtype=torch.float32),requires_grad=trainable)
        self.truncate=truncate; self.kernel_size=kernel_size
        self.padding_mode=padding_mode; self.padding_value=padding_value
    def forward(self,x):
        for d in self.dims:
            x = gaussian_filter_1d(x,dim=d,sigma=self.sigma,truncate=self.truncate,
                                   kernel_size=self.kernel_size,padding_mode=self.padding_mode,
                                   padding_value=self.padding_value)
        return x

class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-12):
        super().__init__()
        self.features=features; self.eps=eps
        self.weight=nn.Parameter(torch.ones(features))
        self.bias=nn.Parameter(torch.zeros(features))
    def forward(self,x):
        h,w=x.shape[2],x.shape[3]
        weight=self.weight.view(-1,1,1).expand(-1,h,w).contiguous()
        bias=self.bias.view(-1,1,1).expand(-1,h,w).contiguous()
        return F.layer_norm(x,(self.features,h,w),weight,bias,self.eps)

class Bias(nn.Module):
    def __init__(self,channels):
        super().__init__(); self.bias=nn.Parameter(torch.zeros(channels))
    def forward(self,x): return x + self.bias[None,:,None,None]

class LayerNormMultiInput(nn.Module):
    def __init__(self,features,eps=1e-12):
        super().__init__(); self.features=features
        for k,f in enumerate(features):
            if f: setattr(self,f'ln{k}',LayerNorm(f,eps=eps))
    def forward(self,tensors):
        return [getattr(self,f'ln{k}')(t) if f else None
                for k,(f,t) in enumerate(zip(self.features,tensors))]

class Conv2dMultiInput(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size,bias=True):
        super().__init__(); self.in_channels=in_channels
        for k,c in enumerate(in_channels):
            if c: setattr(self,f'conv{k}',nn.Conv2d(c,out_channels,kernel_size,bias=bias))
    def forward(self,tensors):
        out=None
        for k,(c,t) in enumerate(zip(self.in_channels,tensors)):
            if not c: continue
            _o=getattr(self,f'conv{k}')(t); out=_o if out is None else out+_o
        return out

print('✅ Cell 3 OK')

✅ Cell 3 OK


In [6]:
import torchvision
from torch.utils import model_zoo

class Normalizer(nn.Module):
    def __init__(self):
        super().__init__()
        m=torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        s=torch.tensor([0.229,0.224,0.225]).view(3,1,1)
        self.register_buffer('mean',m,persistent=False)
        self.register_buffer('std',s,persistent=False)
    def forward(self,x): return (x/255.0-self.mean)/self.std

def _load_shapenet_c():
    url=('https://bitbucket.org/robert_geirhos/texture-vs-shape-pretrained-models'
         '/raw/60b770e128fffcbd8562a3ab3546c1a735432d03'
         '/resnet50_finetune_60_epochs_lr_decay_after_30_start_resnet50'
         '_train_45_epochs_combined_IN_SF-ca06340c.pth.tar')
    model=torchvision.models.resnet50(pretrained=False)
    model=nn.Sequential(OrderedDict([('module',model)]))
    ckpt=model_zoo.load_url(url,map_location='cpu')
    model.load_state_dict(ckpt['state_dict']); return model

class RGBShapeNetC(nn.Sequential):
    def __init__(self): super().__init__(Normalizer(), _load_shapenet_c())
class RGBEfficientNetB5(nn.Sequential):
    def __init__(self):
        from efficientnet_pytorch import EfficientNet
        super().__init__(Normalizer(), EfficientNet.from_pretrained('efficientnet-b5'))
class RGBDenseNet201(nn.Sequential):
    def __init__(self):
        dn=torch.hub.load('pytorch/vision:v0.10.0','densenet201',pretrained=True)
        super().__init__(Normalizer(), dn)
class RGBResNext50(nn.Sequential):
    def __init__(self):
        rx=torch.hub.load('pytorch/vision:v0.10.0','resnext50_32x4d',pretrained=True)
        super().__init__(Normalizer(), rx)

print('✅ Cell 4 OK')

✅ Cell 4 OK


In [7]:
# --- Core modules (Cell 6 gốc) ---
class FeatureExtractor(nn.Module):
    def __init__(self, features, targets):
        super().__init__(); self.features=features; self.targets=targets; self.outputs={}
        for t in targets:
            dict([*self.features.named_modules()])[t].register_forward_hook(self._hook(t))
    def _hook(self,lid):
        def fn(_,__,out): self.outputs[lid]=out.clone()
        return fn
    def forward(self,x):
        self.outputs.clear(); self.features(x)
        return [self.outputs[t] for t in self.targets]

class Finalizer(nn.Module):
    def __init__(self,sigma=8.0,learn_sigma=True,center_bias_weight=1.0,
                 learn_center_bias_weight=True,saliency_map_factor=2):
        super().__init__(); self.smf=saliency_map_factor
        self.gauss=GaussianFilterNd([2,3],sigma,truncate=3,trainable=learn_sigma)
        self.cbw=nn.Parameter(torch.tensor([center_bias_weight]),
                              requires_grad=learn_center_bias_weight)
    def forward(self,readout,centerbias):
        cb=F.interpolate(centerbias.unsqueeze(1),scale_factor=1/self.smf,
                         recompute_scale_factor=False)[:,0]
        out=F.interpolate(readout,size=[cb.shape[1],cb.shape[2]])
        out=self.gauss(out); out=out[:,0]; out=out+self.cbw*cb
        out=F.interpolate(out.unsqueeze(1),
                          size=[centerbias.shape[1],centerbias.shape[2]])[:,0]
        return out - out.logsumexp(dim=(1,2),keepdim=True)

class DeepGazeIIIMixture(nn.Module):
    def __init__(self,features,saliency_networks,scanpath_networks,
                 fixation_selection_networks,finalizers,downsample=2,
                 readout_factor=16,saliency_map_factor=2,included_fixations=None):
        super().__init__()
        self.downsample=downsample; self.rf=readout_factor; self.smf=saliency_map_factor
        self.features=features
        for p in self.features.parameters(): p.requires_grad=False
        self.features.eval()
        self.sal_nets=nn.ModuleList(saliency_networks)
        self.scn_nets=nn.ModuleList(scanpath_networks)
        self.fix_nets=nn.ModuleList(fixation_selection_networks)
        self.fins=nn.ModuleList(finalizers)
    def forward(self,x,centerbias,**kw):
        orig=x.shape
        xd=F.interpolate(x,scale_factor=1/self.downsample,recompute_scale_factor=False)
        with torch.no_grad(): feats=self.features(xd)
        rs=[math.ceil(orig[2]/self.downsample/self.rf),
            math.ceil(orig[3]/self.downsample/self.rf)]
        ri=torch.cat([F.interpolate(f,rs) for f in feats],dim=1)
        preds=[]
        for sal,scn,fix,fin in zip(self.sal_nets,self.scn_nets,self.fix_nets,self.fins):
            s=sal(ri); s=fix((s,None)); s=fin(s,centerbias); preds.append(s.unsqueeze(1))
        preds=torch.cat(preds,dim=1)-np.log(len(self.sal_nets))
        return preds.logsumexp(dim=1,keepdim=True)
    def train(self,mode=True):
        self.features.eval()
        for m in [*self.sal_nets,*self.fix_nets,*self.fins]: m.train(mode)

class MixtureModel(nn.Module):
    def __init__(self,models): super().__init__(); self.models=nn.ModuleList(models)
    def forward(self,x,centerbias,**kw):
        preds=torch.cat([m(x,centerbias) for m in self.models],dim=1)
        preds-=np.log(len(self.models))
        return preds.logsumexp(dim=1,keepdim=True)

# --- Build (Cell 7 gốc) ---
BACKBONES_CFG=[
    {'name':'ShapeNetC','class':RGBShapeNetC,
     'used_features':['1.module.layer3.0.conv2','1.module.layer3.3.conv2',
                      '1.module.layer3.5.conv1','1.module.layer3.5.conv2',
                      '1.module.layer4.1.conv2','1.module.layer4.2.conv2'],'channels':2048},
    {'name':'EfficientNetB5','class':RGBEfficientNetB5,
     'used_features':['1._blocks.24._depthwise_conv','1._blocks.26._depthwise_conv',
                      '1._blocks.35._project_conv'],'channels':2416},
    {'name':'DenseNet201','class':RGBDenseNet201,
     'used_features':['1.features.denseblock4.denselayer32.norm1',
                      '1.features.denseblock4.denselayer32.conv1',
                      '1.features.denseblock4.denselayer31.conv2'],'channels':2048},
    {'name':'ResNext50','class':RGBResNext50,
     'used_features':['1.layer3.5.conv1','1.layer3.5.conv2',
                      '1.layer3.4.conv2','1.layer4.2.conv2'],'channels':2560},
]

def build_saliency_network(C):
    return nn.Sequential(OrderedDict([
        ('layernorm0',LayerNorm(C)),
        ('conv0',nn.Conv2d(C,8,(1,1),bias=False)),('bias0',Bias(8)),('softplus0',nn.Softplus()),
        ('layernorm1',LayerNorm(8)),
        ('conv1',nn.Conv2d(8,16,(1,1),bias=False)),('bias1',Bias(16)),('softplus1',nn.Softplus()),
        ('layernorm2',LayerNorm(16)),
        ('conv2',nn.Conv2d(16,1,(1,1),bias=False)),('bias2',Bias(1)),('softplus3',nn.Softplus()),
    ]))

def build_fixation_selection_network():
    return nn.Sequential(OrderedDict([
        ('layernorm0',LayerNormMultiInput([1,0])),
        ('conv0',Conv2dMultiInput([1,0],128,(1,1),bias=False)),('bias0',Bias(128)),
        ('softplus0',nn.Softplus()),
        ('layernorm1',LayerNorm(128)),
        ('conv1',nn.Conv2d(128,16,(1,1),bias=False)),('bias1',Bias(16)),('softplus1',nn.Softplus()),
        ('conv2',nn.Conv2d(16,1,(1,1),bias=False)),
    ]))

def build_backbone_model(cfg,n):
    extractor=FeatureExtractor(cfg['class'](),cfg['used_features']); C=cfg['channels']
    return DeepGazeIIIMixture(
        features=extractor,
        saliency_networks=[build_saliency_network(C) for _ in range(n)],
        scanpath_networks=[None]*n,
        fixation_selection_networks=[build_fixation_selection_network() for _ in range(n)],
        finalizers=[Finalizer(sigma=CFG['initial_sigma'],learn_sigma=True,
                              saliency_map_factor=CFG['saliency_map_factor']) for _ in range(n)],
        downsample=CFG['downsample'],readout_factor=CFG['readout_factor'],
        saliency_map_factor=CFG['saliency_map_factor'],included_fixations=[])

def build_deepgaze_iie(backbone_indices=None,n_components=None):
    n=n_components or CFG['n_components']
    cfgs=BACKBONES_CFG if backbone_indices is None else [BACKBONES_CFG[i] for i in backbone_indices]
    print('Backbones:',[c['name'] for c in cfgs],'| n_components:',n)
    models=[]
    for cfg in cfgs:
        print(f'  Loading {cfg["name"]}...',end=' ',flush=True)
        models.append(build_backbone_model(cfg,n)); print('✅')
    return MixtureModel(models)

print('✅ Cell 5 OK')

✅ Cell 5 OK


In [8]:
# ============================================================
# CELL 6: METRICS + LOAD CHECKPOINT
# ============================================================

def log_likelihood(log_density, fixation_mask, weights=None):
    if log_density.dim()==4: log_density=log_density[:,0]
    if weights is None: weights=torch.ones(log_density.shape[0],device=log_density.device)
    weights=len(weights)*weights.view(-1,1,1)/weights.sum()
    fix_count=fixation_mask.sum(dim=(-1,-2),keepdim=True).clamp(min=1)
    return torch.mean(weights*torch.sum(log_density*fixation_mask,
                      dim=(-1,-2),keepdim=True)/fix_count)

def nss(log_density, fixation_mask, weights=None):
    if log_density.dim()==4: log_density=log_density[:,0]
    if weights is None: weights=torch.ones(log_density.shape[0],device=log_density.device)
    weights=len(weights)*weights.view(-1,1,1)/weights.sum()
    binary_mask=(fixation_mask>0).float()
    fix_count=binary_mask.sum(dim=(-1,-2),keepdim=True).clamp(min=1)
    density=torch.exp(log_density)
    mean=density.mean(dim=(-1,-2),keepdim=True); std=density.std(dim=(-1,-2),keepdim=True)
    sal_map=(density-mean)/(std+1e-8)
    nss_pi=torch.sum(sal_map*binary_mask,dim=(-1,-2),keepdim=True)/fix_count
    return torch.mean(weights*nss_pi)

def compute_baseline_ll(centerbias_template, fix_data, img_paths, downsample):
    from scipy.ndimage import zoom
    from scipy.special import logsumexp
    from PIL import Image
    lls=[]
    for fp,fix in zip(img_paths,fix_data):
        try: w,h=Image.open(fp).size
        except Exception: continue
        h_=int(h/downsample); w_=int(w/downsample)
        cb=zoom(centerbias_template,
                (h_/centerbias_template.shape[0],w_/centerbias_template.shape[1]),
                order=0,mode='nearest')
        cb-=logsumexp(cb)
        xs=np.array(fix['x'],dtype=float); ys=np.array(fix['y'],dtype=float)
        valid=~(np.isnan(xs)|np.isnan(ys)); xs,ys=xs[valid],ys[valid]
        xs=np.clip((xs/downsample).astype(int),0,w_-1)
        ys=np.clip((ys/downsample).astype(int),0,h_-1)
        if len(xs): lls.append(cb[ys,xs].mean())
    return float(np.mean(lls)) if lls else 0.0

@torch.no_grad()
def evaluate(model, loader, baseline_ll, device):
    model.eval(); lls,nsss,wts=[],[],[]
    from tqdm.notebook import tqdm
    for batch in tqdm(loader,desc='Eval',leave=False):
        img=batch['image'].to(device); cb=batch['centerbias'].to(device)
        mask=batch['fixation_mask'].to(device); w=batch['weight'].to(device)
        out=model(img,cb)
        lls.append(log_likelihood(out,mask,weights=w).item())
        nsss.append(nss(out,mask,weights=w).item())
        wts.append(w.sum().item())
    avg_ll=np.average(lls,weights=wts); avg_nss=np.average(nsss,weights=wts)
    ig_nats=avg_ll-baseline_ll
    return {'LL':avg_ll,'IG_nats':ig_nats,'IG_bits':ig_nats/np.log(2),'NSS':avg_nss}

def load_checkpoint(model, name):
    lp=os.path.join(CFG['local_ckpt'],name); dp=os.path.join(CFG['drive_ckpt'],name)
    path=lp if os.path.exists(lp) else (dp if os.path.exists(dp) else None)
    if path is None: print('❌ Không tìm thấy',name); return 0,{}
    ckpt=torch.load(path,map_location=DEVICE,weights_only=False)
    model.load_state_dict(ckpt['model'])
    print(f'✅ Loaded {name} (epoch {ckpt["epoch"]})')
    return ckpt['epoch'],ckpt.get('metrics',{})

print('✅ Cell 6 OK')

✅ Cell 6 OK


In [9]:
# ============================================================
# CELL 7: CENTER-BIAS + DATASET OOD (tự chứa)
# ============================================================
import urllib.request
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# Tải centerbias
CB_PATH='/content/centerbias_mit1003.npy'
if not os.path.exists(CB_PATH):
    urllib.request.urlretrieve(
        'https://github.com/matthias-k/DeepGaze/releases/download/v1.0.0/centerbias_mit1003.npy',
        CB_PATH)
centerbias_template=np.load(CB_PATH)
print('Centerbias:',centerbias_template.shape)

class OODSaliencyDataset(Dataset):
    """Ảnh + fixation → batch {image, centerbias, fixation_mask, weight}."""
    def __init__(self, img_paths, fix_data, cb_template, downsample):
        self.img_paths=img_paths; self.fix_data=fix_data
        self.cb=cb_template; self.ds=downsample
    def __len__(self): return len(self.img_paths)
    def __getitem__(self, i):
        from scipy.ndimage import zoom
        from scipy.special import logsumexp
        img=Image.open(self.img_paths[i]).convert('RGB')
        w,h=img.size; h_=int(h/self.ds); w_=int(w/self.ds)
        img=img.resize((w_,h_))
        arr=np.array(img,dtype=np.float32).transpose(2,0,1)  # (3,H,W) thang [0,255]
        cb=zoom(self.cb,(h_/self.cb.shape[0],w_/self.cb.shape[1]),order=0,mode='nearest')
        cb=cb-logsumexp(cb)
        xs=np.array(self.fix_data[i]['x'],dtype=float)
        ys=np.array(self.fix_data[i]['y'],dtype=float)
        valid=~(np.isnan(xs)|np.isnan(ys)); xs,ys=xs[valid],ys[valid]
        xs=np.clip((xs/self.ds).astype(int),0,w_-1)
        ys=np.clip((ys/self.ds).astype(int),0,h_-1)
        mask=np.zeros((h_,w_),dtype=np.float32)
        mask[ys,xs]=1.0                       # BINARY mask (đúng cho NSS)
        return {'image':torch.from_numpy(arr),
                'centerbias':torch.from_numpy(cb.astype(np.float32)),
                'fixation_mask':torch.from_numpy(mask),
                'weight':torch.tensor(float(len(xs)))}

def collate_fn(batch):
    # Ảnh OOD cùng dataset thường cùng kích thước → pad về max cho an toàn
    Hs=[b['image'].shape[1] for b in batch]; Ws=[b['image'].shape[2] for b in batch]
    H,W=max(Hs),max(Ws)
    def pad(t,val=0):
        ph,pw=H-t.shape[-2],W-t.shape[-1]
        return F.pad(t,(0,pw,0,ph),value=val)
    imgs=torch.stack([pad(b['image']) for b in batch])
    cbs =torch.stack([pad(b['centerbias'],val=-30) for b in batch])  # log-density rất nhỏ
    masks=torch.stack([pad(b['fixation_mask']) for b in batch])
    ws=torch.stack([b['weight'] for b in batch])
    return {'image':imgs,'centerbias':cbs,'fixation_mask':masks,'weight':ws}

print('✅ Cell 7 OK')

Centerbias: (1024, 1024)
✅ Cell 7 OK


In [10]:
import pysaliency.external_datasets as ed
cands=[x for x in dir(ed) if any(k in x.lower() for k in ['toronto','pascal'])]
print('Hàm tải khả dụng:', cands)

Hàm tải khả dụng: ['get_PASCAL_S', 'get_toronto', 'get_toronto_with_subjects', 'pascal_s', 'toronto']


In [12]:
# ── PATCH: pysaliency dùng np.string_ (bị xóa ở NumPy 2.0) ──
import numpy as np
if not hasattr(np, 'string_'):
    np.string_ = np.bytes_
    print('✅ Đã vá np.string_ → np.bytes_')
else:
    print('np.string_ vẫn còn, không cần vá')

✅ Đã vá np.string_ → np.bytes_


In [13]:
# ============================================================
# CELL 9: TẢI Toronto & PASCAL-S + ĐÁNH GIÁ OOD
# Mô hình đủ 4 backbone, load best_mit1003.pth, KHÔNG train lại
# ============================================================
import pysaliency.external_datasets as ed

OOD_DIR='/content/ood_datasets'; os.makedirs(OOD_DIR,exist_ok=True)

def pysaliency_to_pipeline(stimuli, fixations):
    """stimuli+fixations (pysaliency) → img_paths + [{'x','y'}]."""
    img_paths, fix_data = [], []
    for n in range(len(stimuli)):
        try:
            p = stimuli.filenames[n]
        except Exception:
            p = stimuli.stimuli[n] if hasattr(stimuli,'stimuli') else None
        m = fixations.n == n
        xs, ys = fixations.x[m], fixations.y[m]
        if p is not None and len(xs):
            img_paths.append(p)
            fix_data.append({'x':xs.tolist(),'y':ys.tolist()})
    return img_paths, fix_data

def evaluate_ood(name, imgs, fixd, downsample):
    baseline=compute_baseline_ll(centerbias_template, fixd, imgs, downsample)
    dl=DataLoader(OODSaliencyDataset(imgs,fixd,centerbias_template,downsample),
                  batch_size=CFG['batch_size'],shuffle=False,
                  num_workers=CFG['num_workers'],collate_fn=collate_fn)
    mets=evaluate(model, dl, baseline, DEVICE)
    print(f'\n═══ {name} (OOD) ═══')
    print(f'  Ảnh: {len(imgs)} | Baseline: {baseline:.4f} nats')
    print(f'  IG : {mets["IG_nats"]:.4f} nats = {mets["IG_bits"]:.4f} bits')
    print(f'  NSS: {mets["NSS"]:.4f}')
    return {'dataset':name, **mets}

# ── Dựng model đủ 4 backbone + load checkpoint ────────────
print('Đang dựng model 4 backbone (tải backbone, mất vài phút)...')
model = build_deepgaze_iie(backbone_indices=None, n_components=CFG['n_components']).to(DEVICE)
load_checkpoint(model, 'best_mit1003.pth')
model.eval()

results_ood=[]

# ── Toronto ───────────────────────────────────────────────
print('\nTải Toronto...')
try:
    s,f = ed.get_toronto(location=OOD_DIR)
    i,fd = pysaliency_to_pipeline(s,f)
    print(f'Toronto: {len(i)} ảnh có fixation')
    results_ood.append(evaluate_ood('Toronto', i, fd, CFG['ds_mit']))
except Exception as e:
    import traceback; traceback.print_exc()
    print(f'❌ Toronto lỗi → nếu BadZipFile, chạy lại cell')

# ── PASCAL-S ──────────────────────────────────────────────
print('\nTải PASCAL-S...')
try:
    s,f = ed.get_PASCAL_S(location=OOD_DIR)
    i,fd = pysaliency_to_pipeline(s,f)
    print(f'PASCAL-S: {len(i)} ảnh có fixation')
    results_ood.append(evaluate_ood('PASCAL-S', i, fd, CFG['ds_mit']))
except Exception as e:
    import traceback; traceback.print_exc()
    print(f'❌ PASCAL-S lỗi → nếu BadZipFile, chạy lại cell')

# ── Bảng tổng hợp ─────────────────────────────────────────
if results_ood:
    print('\n'+'='*52)
    print(f'{"Dataset":<12}{"IG(nats)":>12}{"IG(bits)":>12}{"NSS":>10}')
    print('='*52)
    for r in results_ood:
        print(f'{r["dataset"]:<12}{r["IG_nats"]:>12.4f}{r["IG_bits"]:>12.4f}{r["NSS"]:>10.4f}')
    print('='*52)

print('\n✅ Hoàn tất OOD')

Đang dựng model 4 backbone (tải backbone, mất vài phút)...
Backbones: ['ShapeNetC', 'EfficientNetB5', 'DenseNet201', 'ResNext50'] | n_components: 10
  Loading ShapeNetC... ✅
  Loading EfficientNetB5... Loaded pretrained weights for efficientnet-b5
✅
  Loading DenseNet201... 

Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.10.0


✅
  Loading ResNext50... 

Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.10.0


✅
✅ Loaded best_mit1003.pth (epoch 3)

Tải Toronto...


...

Eval:   0%|          | 0/60 [00:00<?, ?it/s]


═══ Toronto (OOD) ═══
  Ảnh: 120 | Baseline: -10.7573 nats
  IG : 0.4639 nats = 0.6693 bits
  NSS: 2.0876

Tải PASCAL-S...


Checking md5 sum...
Creating stimuli
Creating fixations
PASCAL-S: 850 ảnh có fixation


Eval:   0%|          | 0/425 [00:00<?, ?it/s]


═══ PASCAL-S (OOD) ═══
  Ảnh: 850 | Baseline: -9.7750 nats
  IG : 0.3951 nats = 0.5700 bits
  NSS: 2.4861

Dataset         IG(nats)    IG(bits)       NSS
Toronto           0.4639      0.6693    2.0876
PASCAL-S          0.3951      0.5700    2.4861

✅ Hoàn tất OOD


In [14]:
import json
with open('/content/drive/MyDrive/Data PBL4/ood_results.json','w') as f:
    json.dump(results_ood, f, indent=2)
print('✅ Đã lưu kết quả OOD ra Drive')

✅ Đã lưu kết quả OOD ra Drive
